# Option chains

Written against [`ib_async`](https://github.com/ib-api-reloaded/ib_async) itself,
unmodified: `ib_async_dx.IB` is its `IB` with the engine in place of the one
layer of it that expects a socket to a gateway.

The options on the S&P 500 index: the next three monthly expiries, at strikes
within 20 points of the index, on multiples of 5.

## Connecting

`IB.connect` was written for a gateway, so it takes a host, a port and a client
id. Here it needs none of them: it takes the credentials instead, and there is
no local process to reach.

`ib.sleep()` rather than `time.sleep()` throughout. The library's loop runs on
this thread, and a plain sleep stops it — every stream then reads as dead.

In [ ]:
import os
from dotenv import load_dotenv
from ib_async_dx import IB, util

util.startLoop()
load_dotenv()

ib = IB()
ib.connect(
    username=os.environ["IB_USERNAME"],
    password=os.environ["IB_PASSWORD"],
    paper=True,
)

print(f"connected: {ib.isConnected()}")
print(f"accounts:  {ib.managedAccounts()}")

## The underlying

Qualify the index first: ib_async keys a `Ticker` by the contract, and a
contract without its `conId` cannot be a key. Delayed frozen data, market data
type 4, so the notebook runs on an account without a subscription to the index.

In [ ]:
from ib_async_dx import Index

spx = Index("SPX", "CBOE")
ib.qualifyContracts(spx)

ib.reqMarketDataType(4)
[ticker] = ib.reqTickers(spx)
spxValue = ticker.marketPrice()
print(f"SPX {spxValue}")

## The chains

One chain per exchange and trading class: `SPX` is the monthly options and
`SPXW` the weeklies, whose expiries are disjoint from the monthlies'.

In [ ]:
chains = ib.reqSecDefOptParams(spx.symbol, "", spx.secType, spx.conId)
util.df(chains)

In [ ]:
chain = next(c for c in chains if c.tradingClass == "SPX" and c.exchange == "SMART")
print(f"{len(chain.expirations)} expiries, {len(chain.strikes)} strikes")

## The options

The whole matrix of expiries and strikes is there; these are the ones that
meet the conditions, qualified in one call.

In [ ]:
from ib_async_dx import Option

strikes = [
    strike
    for strike in chain.strikes
    if strike % 5 == 0 and spxValue - 20 < strike < spxValue + 20
]
expirations = sorted(chain.expirations)[:3]

contracts = [
    Option("SPX", expiration, strike, right, "SMART", tradingClass="SPX")
    for right in ("P", "C")
    for expiration in expirations
    for strike in strikes
]
contracts = ib.qualifyContracts(*contracts)
print(f"{len(contracts)} options")

## Their quotes and greeks

Market data for all of them in one go, with the venue's model of each option
in `modelGreeks`.

In [ ]:
tickers = ib.reqTickers(*contracts)

for t in tickers[:6]:
    g = t.modelGreeks
    print(f"{t.contract.localSymbol:22} bid {t.bid:>8}  ask {t.ask:>8}  "
          f"iv {g.impliedVol if g else None}  delta {g.delta if g else None}")

In [ ]:
ib.disconnect()